# Step 1 — Transition-order diagnostics on an exact Gaussian-mixture control

**Purpose.** Before touching any trained model, we validate every diagnostic on a system whose transition order we know analytically.

- Data: two classes with means `+s` (weight `w`) and `-s` (weight `1-w`), `s` a fixed ±1 pattern in `R^d`.
- Forward (variance-exploding) noising: `x_t = x0 + sigma * eps`.
- Order parameter: `m = (s · x_t) / d`, which is exactly `w N(+1, tau) + (1-w) N(-1, tau)` with **`tau = sigma^2 / d`**.

**Known answers.**
- `w = 0.5` → **pitchfork** (continuous): modes split from 0 at `tau = 1`, branches grow like `sqrt(3(1 - tau))`.
- `w = 0.8` → **saddle-node** (first-order-like): the minor mode appears at a *finite* |m| (≈ 0.74) at `tau_sn ≈ 0.45`.

**Diagnostics.** (A) fixed points of the reverse drift = stationary points of `log p_tau(m)`; (B) mode count; (C) Binder cumulant `U4`; (D) variance across samples of the class-posterior entropy `H[c|m]` (the "heat capacity" analogue).

Runs on CPU in ~2 minutes. No accelerator needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
GRID = np.linspace(-4.0, 4.0, 8001)
plt.rcParams["figure.dpi"] = 110

## Core functions

In [ ]:
def mixture_pdf(m, tau, w):
    g = lambda mu: np.exp(-(m - mu) ** 2 / (2 * tau)) / np.sqrt(2 * np.pi * tau)
    return w * g(1.0) + (1 - w) * g(-1.0)


def stationary_points(tau, w, grid=GRID):
    """Roots of d/dm log p_tau(m) = fixed points of the reverse drift, labelled max/min."""
    dlogp = np.gradient(np.log(mixture_pdf(grid, tau, w)), grid)
    idx = np.where(np.sign(dlogp[:-1]) != np.sign(dlogp[1:]))[0]
    pts = []
    for i in idx:
        root = grid[i] - dlogp[i] * (grid[i + 1] - grid[i]) / (dlogp[i + 1] - dlogp[i])
        pts.append((float(root), "max" if dlogp[i] > 0 else "min"))
    return pts


def sample_m(n, tau, w):
    c = rng.random(n) < w
    mu = np.where(c, 1.0, -1.0)
    return mu + np.sqrt(tau) * rng.standard_normal(n)


def posterior_entropy(m, tau, w):
    a = np.log(w) - (m - 1) ** 2 / (2 * tau)
    b = np.log(1 - w) - (m + 1) ** 2 / (2 * tau)
    p = 1.0 / (1.0 + np.exp(b - a))
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def binder(m):
    mc = m - m.mean()
    return 1.0 - (mc ** 4).mean() / (3.0 * (mc ** 2).mean() ** 2)


def diagnostics(tau, w, n=200_000):
    m = sample_m(n, tau, w)
    H = posterior_entropy(m, tau, w)
    pts = stationary_points(tau, w)
    return dict(tau=tau,
                n_modes=sum(k == "max" for _, k in pts),
                U4=binder(m),
                varH=H.var(),
                meanH=H.mean(),
                stat=pts)

## Run both cases across tau

In [ ]:
TAUS = np.geomspace(0.05, 5.0, 41)
RESULTS = {}

for w in [0.5, 0.8]:
    rows = [diagnostics(t, w) for t in TAUS]
    RESULTS[w] = rows
    print(f"\n=== w = {w} ===")
    print(f"{'tau':>7} {'modes':>5} {'U4':>8} {'Var[H]':>9} {'E[H]':>7}  stationary points")
    for r in rows:
        st = [(round(p, 3), k) for p, k in r['stat']]
        print(f"{r['tau']:7.3f} {r['n_modes']:5d} {r['U4']:8.4f} {r['varH']:9.5f} {r['meanH']:7.4f}  {st}")
    two = [r['tau'] for r in rows if r['n_modes'] == 2]
    peak = max(rows, key=lambda r: r['varH'])
    print(f"-> last bimodal tau (coarse grid): {max(two):.3f}" if two else "-> never bimodal")
    print(f"-> Var[H] peak at tau = {peak['tau']:.3f}, Var[H] = {peak['varH']:.5f}")

## Bifurcation diagrams (ground truth)

Solid = stable fixed points (maxima of `p_tau(m)`), dashed = unstable (minima).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, w in zip(axes, [0.5, 0.8]):
    taus_fine = np.geomspace(0.05, 3.0, 400)
    for tau in taus_fine:
        for p, k in stationary_points(tau, w):
            ax.plot(tau, p, '.', ms=2, color='C0' if k == 'max' else 'C3')
    ax.set_xscale('log'); ax.set_xlabel('tau = sigma^2 / d'); ax.set_ylabel('fixed point m*')
    ax.set_title(f"w = {w}: {'pitchfork' if w == 0.5 else 'saddle-node'} expected")
    ax.axhline(0, color='k', lw=0.5)
plt.tight_layout(); plt.show()

## Diagnostics vs tau

Vertical lines mark the theoretical bifurcation points (tau = 1 for the pitchfork; tau_sn for the saddle-node, located below).

In [ ]:
def saddle_node_locate(w=0.8):
    lo, hi = 0.05, 1.0
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if len(stationary_points(mid, w)) == 3:
            lo = mid
        else:
            hi = mid
    return lo

TAU_SN = saddle_node_locate(0.8)
pts = stationary_points(TAU_SN, 0.8)
print(f"tau_sn ~ {TAU_SN:.4f}; stationary points just below it: {[(round(p,3),k) for p,k in pts]}")
print("Expected: minor max appears at FINITE |m| (~ -0.74), unlike the pitchfork which grows from 0.")

print("\nPitchfork check (w=0.5): m* vs sqrt(3(1-tau))")
for tau in [0.90, 0.95, 0.98, 0.99]:
    ms = [p for p, k in stationary_points(tau, 0.5) if k == 'max' and p > 0]
    print(f"tau={tau:.2f}  m*={ms[0] if ms else float('nan'):.4f}  approx={np.sqrt(3*(1-tau)):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for w, col in [(0.5, 'C0'), (0.8, 'C1')]:
    rows = RESULTS[w]
    t = [r['tau'] for r in rows]
    axes[0].plot(t, [r['n_modes'] for r in rows], 'o-', color=col, label=f'w={w}')
    axes[1].plot(t, [r['U4'] for r in rows], 'o-', color=col, label=f'w={w}')
    axes[2].plot(t, [r['varH'] for r in rows], 'o-', color=col, label=f'w={w}')
for ax, name in zip(axes, ['number of modes', 'Binder U4 (centered m)', 'Var[H(c|m)]  (heat-capacity proxy)']):
    ax.set_xscale('log'); ax.set_xlabel('tau'); ax.set_title(name)
    ax.axvline(1.0, color='C0', ls='--', lw=0.8); ax.axvline(TAU_SN, color='C1', ls='--', lw=0.8)
    ax.legend()
plt.tight_layout(); plt.show()

## d-independence check

Full d-dimensional simulation. At fixed tau, U4 and Var[H] must not depend on d: this is the mean-field collapse that makes finite-size scaling *trivial* in this control, and is why Phase 2 needs data with local structure.

In [ ]:
w, tau, n = 0.5, 0.7, 100_000
for d in [64, 256, 1024]:
    s = np.sign(rng.standard_normal(d))
    c = rng.random(n) < w
    x0 = np.where(c[:, None], s, -s)
    sigma = np.sqrt(tau * d)
    xt = x0 + sigma * rng.standard_normal((n, d))
    m = xt @ s / d
    H = posterior_entropy(m, tau, w)
    print(f"d={d:5d}  U4={binder(m):.4f}  Var[H]={H.var():.5f}  sd(m)={m.std():.4f}  (theory {np.sqrt(1+tau):.4f})")

## What to report back

1. Mode-count transition tau for `w=0.5` (expect 1.0) and `w=0.8` (expect `tau_sn` ≈ 0.45).
2. The pitchfork check table.
3. **Where the Var[H] peak sits relative to the bifurcation point in each case.** This is the open question: if the heat-capacity proxy peaks away from the fixed-point bifurcation, the paper needs to explain the offset.
4. The d-independence table (U4 and Var[H] should agree across d to within Monte-Carlo noise).